# Lesson 4.3: Pandas Fundamentals

**Analyst:** Meron Welderufael
**Date:** 10/26/2026
**Client:** FinFlow (Product Analytics engagement led by Hannah Whitford)
**Notebook purpose:** Load the FinFlow synthetic dataset into Pandas, run
the standard inspection routine, and practice basic DataFrame creation
and saving.

This notebook is the second day of the FinFlow Customer Churn Investigation.
The 1,000-row CSV loaded here is the same dataset generated in Lesson 4.2.
By Friday it will be replaced with a 200,000-row version for the full project.

In [1]:
# Imports
import pandas as pd
import numpy as np
from faker import Faker, VERSION
import random
from datetime import date, timedelta
from pathlib import Path
import csv

# Display all DataFrame columns when printing (Pandas truncates by default)
pd.set_option('display.max_columns', None)

# Show numbers with two decimal places where appropriate
pd.set_option('display.float_format', '{:.2f}'.format)

# Set seeds for reproducibility.
# The number 42 is conventional. 

SEED = 42
Faker.seed(SEED)
random.seed(SEED)

# Initialize the Faker instance with US English locale.
# FinFlow's customer base is US-concentrated, so we want US-flavored names,
# addresses, and phone numbers.
fake = Faker('en_US')

# Pin the dataset to a fixed "as of" date.
AS_OF = date(2026, 10, 21)
SIGNUP_WINDOW_DAYS = 730
SIGNUP_WINDOW_START = AS_OF - timedelta(days=SIGNUP_WINDOW_DAYS)

## Step A: Scale the generation script.

In [2]:
# A lookup from plan_tier to monthly_fee.
# These are the locked FinFlow pricing tiers from the kickoff call.abs

PLAN_FEES = {
    'Basic': 9.99,
    'Pro': 29.99,
    'Enterprise': 199.00
}

def build_one_customer_v2(index):
     """Build a single FinFlow customer record, version 2 with correlations.
    Args:
        index: integer counter used to generate sequential customer_id.
    Returns:
        Dictionary with all FinFlow customer fields populated.
    """
     first = fake.first_name()
     last = fake.last_name()
     signup_date = SIGNUP_WINDOW_START + timedelta(
         days = random.triangular(
             low= 0,
             high=SIGNUP_WINDOW_DAYS ,
             mode=SIGNUP_WINDOW_DAYS - 180))
    
      # plan_tier with realistic FinFlow distribution
     plan_tier = random.choices(
        ['Basic', 'Pro', 'Enterprise'],
        weights = [60,30,10],
        k=1)[0]
    
     monthly_fee = PLAN_FEES[plan_tier]
    
     # churn rate with correlate plan_tier
     churn_rates = {
        "Basic": 25,
        "Pro": 15,
        "Enterprise": 5
     }
     churn_rate = churn_rates[plan_tier]
     is_active = random.choices(
        [True, False],
         weights = [100 - churn_rate,churn_rate],
         k=1)[0]
        
    
      #last_login_date and churn_date depend on active state,
     # measured from the pinned as-of date rather than the clock
     today = AS_OF
     churn_date = None
    
     if is_active:
         # Active customers logged in within the last 30 days
         days_since_login = random.randint(0, 30)
         last_login = today - timedelta(days=days_since_login)
     else:
         # Churned customers haven't logged in for 30 to 180 days
         days_since_login = random.randint(30, 180)
         last_login = today - timedelta(days=days_since_login)

         # Churn date is between last_login and today
         days_since_churn = random.randint(0, days_since_login)
         churn_date = today - timedelta(days=days_since_churn)

     # Engagement metrics: correlate with active state
     if is_active:
         total_sessions = random.randint(20, 400)
         avg_session_duration = round(random.uniform(3.0, 15.0), 1)
         support_tickets = random.randint(0, 4)
     else:
         total_sessions = random.randint(1, 50)
         avg_session_duration = round(random.uniform(1.0, 6.0), 1)
         support_tickets = random.randint(0, 8)
         
     referral_source = random.choices(
         ["Organic Search", "Paid Search", "Social Media", "Word of Mouth", "Affiliate", "Direct"],
         weights = [45, 20, 15,10, 7, 3],
         k =1)[0]

     return {
         "customer_id": f"FF-{index:05d}",
         "first_name": first,
         "last_name": last,
         "email": f"{first.lower()}.{last.lower()}{index}@example.com",
         "city": fake.city(),
         "state": fake.state_abbr(),
         "country": "US",
         "signup_date": signup_date.isoformat(),
         "plan_tier": plan_tier,
         "monthly_fee": monthly_fee,
         "last_login_date": last_login.isoformat(),
         "total_sessions": total_sessions,
         "avg_session_duration_minutes": avg_session_duration,
         "support_tickets_opened": support_tickets,
         "active": is_active,
         "churn_date": churn_date.isoformat() if churn_date else None,
         "referral_source": referral_source
     }

In [3]:
import time

# Re-seed so the run is reproducible from this point
Faker.seed(SEED)
random.seed(SEED)

# Generate 200,000 customers
start = time.time()
customers = [build_one_customer_v2(i) for i in range(1,200001)]
elapsed = time.time() - start

print(f'Built {len(customers)} customers in {elapsed:.2f} seconds')
print()
print("First customer:")
for field, value in customers[0].items():
    print(f'{field}: {value}')

Built 200000 customers in 88.39 seconds

First customer:
customer_id: FF-00001
first_name: Danielle
last_name: Johnson
email: danielle.johnson1@example.com
city: Johnberg
state: WA
country: US
signup_date: 2026-03-11
plan_tier: Basic
monthly_fee: 9.99
last_login_date: 2026-10-14
total_sessions: 91
avg_session_duration_minutes: 11.8
support_tickets_opened: 4
active: True
churn_date: None
referral_source: Organic Search


In [4]:
# Define the output path. Path() handles Mac/Windows path differences cleanly.
output_path = Path("finflow-customers-200k.csv")

# Get the field names from the first record (Python 3.7+ dicts preserve order)
field_names = list(customers[0].keys())

# Write the CSV
with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=field_names)
    writer.writeheader()
    writer.writerows(customers)

# Verify
print(f"Wrote {len(customers)} customers to {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size:,} bytes")

Wrote 200000 customers to C:\Users\Lenovo\northstar-coursework\module-04-python-for-data-analytics\lesson-4-3-pandas-fundamentals\finflow-customers-200k.csv
File size: 28,012,876 bytes


## Step B: Load and inspect.

In [5]:
### Loading the FinFlow synthetic dataset: 200k
df = pd.read_csv('finflow-customers-200k.csv',
                parse_dates=['signup_date', 'last_login_date', 'churn_date'])

# Shape: how many rows and columns?
print("Shape:" ,df.shape)

Shape: (200000, 17)


In [6]:
# Data types per column
print(df.dtypes)

customer_id                                str
first_name                                 str
last_name                                  str
email                                      str
city                                       str
state                                      str
country                                    str
signup_date                     datetime64[us]
plan_tier                                  str
monthly_fee                            float64
last_login_date                 datetime64[us]
total_sessions                           int64
avg_session_duration_minutes           float64
support_tickets_opened                   int64
active                                    bool
churn_date                      datetime64[us]
referral_source                            str
dtype: object


In [7]:
# First five rows
df.head()

,customer_id,first_name,last_name,email,city,state,country,signup_date,plan_tier,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,active,churn_date,referral_source
0,FF-00001,Danielle,Johnson,danielle.johnson1@example.com,Johnberg,WA,US,2026-03-11,Basic,9.99,2026-10-14,91,11.80,4,True,NaT,Organic Search
1,FF-00002,Curtis,Yang,curtis.yang2@example.com,New Roberttown,CO,US,2025-12-06,Basic,9.99,2026-10-05,328,3.30,1,True,NaT,Social Media
2,FF-00003,Jeffery,Wagner,jeffery.wagner3@example.com,New Jamesside,MT,US,2026-04-04,Basic,9.99,2026-10-13,23,12.10,1,True,NaT,Social Media
3,FF-00004,Jeffrey,Lawrence,jeffrey.lawrence4@example.com,Port Lindachester,MA,US,2025-10-25,Basic,9.99,2026-06-27,6,2.90,5,False,2026-10-08,Word of Mouth
4,FF-00005,Joshua,Lewis,joshua.lewis5@example.com,Curtisfurt,CT,US,2026-02-25,Pro,29.99,2026-10-04,83,14.70,3,True,NaT,Organic Search


In [8]:
# Last five rows
df.tail()

,customer_id,first_name,last_name,email,city,state,country,signup_date,plan_tier,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,active,churn_date,referral_source
199995,FF-199996,Mitchell,Johnson,mitchell.johnson199996@example.com,North Ronald,KY,US,2025-12-13,Pro,29.99,2026-10-02,54,5.50,1,True,NaT,Paid Search
199996,FF-199997,Crystal,Thompson,crystal.thompson199997@example.com,Carolynborough,MO,US,2025-08-12,Pro,29.99,2026-09-22,159,4.70,3,True,NaT,Social Media
199997,FF-199998,Erik,Wiley,erik.wiley199998@example.com,Jeffreyhaven,KS,US,2025-09-07,Basic,9.99,2026-10-09,400,11.60,3,True,NaT,Affiliate
199998,FF-199999,Jonathan,Arias,jonathan.arias199999@example.com,Kellyfurt,NM,US,2025-11-27,Basic,9.99,2026-10-08,111,10.40,4,True,NaT,Organic Search
199999,FF-200000,Jeffrey,Johnson,jeffrey.johnson200000@example.com,Brownview,WY,US,2026-01-03,Basic,9.99,2026-09-30,94,3.80,4,True,NaT,Organic Search


In [9]:
# Full info dump
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 17 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   customer_id                   200000 non-null  str           
 1   first_name                    200000 non-null  str           
 2   last_name                     200000 non-null  str           
 3   email                         200000 non-null  str           
 4   city                          200000 non-null  str           
 5   state                         200000 non-null  str           
 6   country                       200000 non-null  str           
 7   signup_date                   200000 non-null  datetime64[us]
 8   plan_tier                     200000 non-null  str           
 9   monthly_fee                   200000 non-null  float64       
 10  last_login_date               200000 non-null  datetime64[us]
 11  total_sessions          

In [10]:
# Numeric summary statistics
df.describe()

,signup_date,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,churn_date
count,200000,200000.00,200000,200000.00,200000.00,200000.00,39998
mean,2025-12-21 11:06:28.079999,34.98,2026-09-18 01:18:40.032000,173.27,7.89,2.41,2026-08-29 11:47:58.523926
min,2024-10-22 00:00:00,9.99,2026-04-24 00:00:00,1.00,1.00,0.00,2026-04-24 00:00:00
25%,2025-09-03 00:00:00,9.99,2026-09-22 00:00:00,48.00,4.60,1.00,2026-08-05 00:00:00
50%,2026-01-12 00:00:00,9.99,2026-10-02 00:00:00,163.00,7.50,2.00,2026-09-08 00:00:00
75%,2026-04-22 00:00:00,29.99,2026-10-12 00:00:00,282.00,11.20,4.00,2026-09-30 00:00:00
max,2026-10-19 00:00:00,199.00,2026-10-21 00:00:00,400.00,15.00,8.00,2026-10-21 00:00:00
std,NaN,55.56,NaN,123.19,3.86,1.89,NaN


## Step C: Sanity checks.

In [22]:
# Plan tier distribution
print("Plan tier counts:")
tier_count = df['plan_tier'].value_counts()
print(tier_count)

print("\nPlan tier percentages:")
tier_percentage= (df['plan_tier'].value_counts(normalize=True) * 100).round(2)
print(tier_percentage)

Plan tier counts:
plan_tier
Basic         120016
Pro            59878
Enterprise     20106
Name: count, dtype: int64

Plan tier percentages:
plan_tier
Basic        60.01
Pro          29.94
Enterprise   10.05
Name: proportion, dtype: float64


In [23]:
#churn rate
churn_rate = (df['active'] == False).mean() * 100
print(f'\nChurn rate: {churn_rate:.2f}%')


Churn rate: 20.00%


In [24]:
#Total monthly recurring revenue from active customers
total_mrr = df.loc[df['active']==True, 'monthly_fee'].sum()
print(f"Total MRR from active customers: ${total_mrr:,.2f}")

Total MRR from active customers: $6,229,663.06


In [25]:
#Date range of signup_date
print("\nSignup date range:")
print("Earliest:", df['signup_date'].min())
print("Latest:", df['signup_date'].max())


Signup date range:
Earliest: 2024-10-22 00:00:00
Latest: 2026-10-19 00:00:00


Generating 200,000 customers took 88 seconds, which was 28 seconds longer than my estimate of 60 seconds. The plan tier and churn distributions appear closer to the design targets across many rows. The larger dataset also took noticeable time to generate, and I will check how long the CSV took to load. Overall, this exercise helped me understand how dataset size affects generation and loading performance.